In [0]:
"""
02_product_dimension.py

Product Dimension (SCD Type 2)

Source:
    work_order_events

Target:
    product_dimension

Author:
Sumanth Vempalle

Version:
2.2.0
"""

import dlt

from pyspark.sql.functions import (
    col,
)

# ============================================================
# Product Source View
# ============================================================

@dlt.view(
    name="product_dimension_source",
    comment="Source view for Product Dimension."
)
def product_dimension_source():

    return (

        spark.readStream.table(
            "work_order_events"
        )

        .select(

            col("product_code"),

            col("product_name"),

            col("family"),

            col("rated_voltage_kv"),

            col("routing_version"),

            col("event_timestamp")
                .alias("last_updated"),

        )

        .dropDuplicates(
            ["product_code", "last_updated"]
        )

    )


# ============================================================
# Target Streaming Table
# ============================================================

dlt.create_streaming_table(

    name="product_dimension",

    comment="Product Dimension (SCD Type 2)."

)


# ============================================================
# AUTO CDC FLOW
# ============================================================

dlt.create_auto_cdc_flow(

    target="product_dimension",

    source="product_dimension_source",

    keys=[
        "product_code",
    ],

    sequence_by="last_updated",

    stored_as_scd_type=2,

    track_history_except_column_list=[
        "last_updated",
    ],

)